# Lecture 9 — Introduction to Streamlit 
## From Charts to Dashboards

> **Dataset:** World Happiness Report 2023

> **Big Book of Dashboards:** Introduction + Chapter 1 — What is a dashboard?

---
## Opening: Model Answer Review

---
## What Is [Streamlit](https://streamlit.io/)?

**Streamlit** is an **open-source Python library** for building **interactive web apps and dashboards** using only Python — no HTML, CSS, or JavaScript required. You write Python code (just like a normal script) and Streamlit automatically generates a web interface with charts, widgets (sliders, buttons, dropdowns), tables, and more. It’s designed especially for **data scientists**, **analysts**, and **machine learning engineers** who want to share insights and tools quickly. 

Key features:

- **Python-centric development:** build full web apps without frontend languages.
- **Interactive widgets:** users can interact with the app in real time.
- **Automatic re-execution:** the app reruns when inputs change, updating the UI instantly.
- **Easy deployment:** publish to Streamlit Community Cloud or other hosting platforms with minimal configuration.

**[Streamlit Sheet Cheet](https://docs.streamlit.io/develop/quick-reference/cheat-sheet)**

Streamlit is not only a teaching or prototyping tool — it is widely used in **industry** for internal analytics tools, decision-support dashboards, and machine-learning applications. For example, in **Snowflake**, a leading cloud data platform, uses Streamlit extensively for **internal dashboards and data tools** across teams. Employees build Streamlit apps to explore metrics, monitor systems, and support business and engineering decisions — without relying on separate frontend teams. Snowflake reports having **hundreds of Streamlit apps** running internally.

---
## What Is a Dashboard?


### What is a dashboard: a visual display of data used to monitor conditions and/or facilitate understanding.

### Two things matter in this definition:

1. It must be visual — numbers in a table are not a dashboard
2. It must have a purpose — monitoring OR understanding. If you cannot say which of these your dashboard does, you do not yet have a dashboard idea, you have a data collection

### Chart vs Dashboard — the key difference

| | Chart | Dashboard |
|---|---|---|
| Answers | One question | Multiple related questions simultaneously |
| Audience | Anyone | A specific person or role |
| Usage | Once, to make a point | Repeatedly, to monitor over time |
| Filters | None to minimal | User-controlled |
| Success measure | Did the audience understand the message? | Does the audience return to use it? |


### 💡 The single-purpose rule

Before any code, write this sentence:

> ❌ *'This dashboard shows the World Happiness data.'* (topic, not purpose)

> ✅ *'This dashboard helps a UN policy analyst see which countries lead on happiness and what factors drive the gap.'* (audience + purpose)

---
## Let's Code Some Examples 💻 

> **HOW TO RUN:** Copy each block into `app.py`, run `streamlit run app.py`.


In [1]:
import streamlit as st
import pandas as pd

In [2]:
df = pd.read_csv('../data/world_happiness_2023.csv')

df.head()

,Country name,Regional indicator,Ladder score,Logged GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [ ]:
# ── STEP 1: Minimal working app ──────────────────────────────────────────
import streamlit as st
import pandas as pd


df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

st.set_page_config(page_title="World Happiness Report", page_icon="🌐")

st.title("World Happiness Dashboard")
st.write(f"Data loaded: {len(df)} countries")


In [ ]:
# ── STEP 2: Add a chart  ───────────────────────────

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="World Happiness Report", page_icon="🌐")

df = pd.read_csv('../data/world_happiness_2023.csv')

df.columns = ['Country','Region','Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

st.title("World Happiness Dashboard")
st.write(f"Data loaded: {len(df)} countries")

# add a bar chart of top 15 countries in Happiness Score
top15 = df.nlargest(15, 'Score').sort_values('Score')
fig = px.bar(top15, x='Score', y='Country', orientation='h',
             labels={'Score': 'Happiness Score (0–10)', 'Country': ''},
             template='plotly_white', title="Happiest 15 Countries in 2023")

fig.update_layout(title_font=dict(family='Arial', size=13, color='slategrey'),
                  xaxis=dict(range=[0, 8.0]))


st.plotly_chart(fig, width='stretch') 


In [ ]:
# ── STEP 3: Sidebar filter + reactive chart ──────────────────────────────

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="World Happiness Report", page_icon="🌐")

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

st.title("World Happiness Dashboard 🌐")

# Add a filter for region and a filter for number of topc countries to show
with st.sidebar:
    st.header("Filters")
    regions = ['All Regions'] + sorted(df['Region'].unique().tolist())
    selected_region = st.selectbox("Region", regions)
    top_n = st.slider("Show top N countries", 5, 20, 10)

# filter data accordingly: 
filtered = df if selected_region == 'All Regions' else df[df['Region'] == selected_region]
top = filtered.nlargest(top_n, 'Score').sort_values('Score')

# create top bar chart
fig = px.bar(top, x='Score', y='Country', orientation='h',
             labels={'Score': 'Happiness Score (0–10)', 'Country': ''},
             template='plotly_white', title=f'Happiest {top_n} Countries in 2023 - {selected_region}')

fig.update_layout(title_font=dict(family='Arial', size=13, color='slategrey'),
                  xaxis=dict(range=[0, 8.0]))


st.plotly_chart(fig, width='stretch', height=600) 

In [ ]:
# ── STEP 4: Full dashboard — KPIs + Sidebar Filters + reactive chart ──────────


import streamlit as st
import pandas as pd
import plotly.express as px

# Page configuration, title and data source
st.set_page_config(page_title="World Happiness Report", page_icon="🌐", layout='wide')
st.title("🌍 World Happiness Dashboard")
st.caption("Source: World Happiness Report 2023 | Kaggle")

# Data Ingestion
df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

# Side bar filters
with st.sidebar:
    st.header("Filters")
    regions = ['All Regions'] + sorted(df['Region'].unique().tolist())
    selected_region = st.selectbox("Region", regions)
    top_n = st.slider("Show top N", 5, 25, 15)
    

# filter data by region and top_n
filtered = df if selected_region == 'All Regions' else df[df['Region'] == selected_region]
top = filtered.nlargest(top_n, 'Score').sort_values('Score')

# KPI row 
col1, col2, col3 = st.columns(3)
col1.metric("Countries", len(filtered))
col2.metric("Avg Score", f"{filtered['Score'].mean():.2f}",
            f"{filtered['Score'].mean()-df['Score'].mean():+.2f} vs global")
col3.metric("Happiest", filtered.nlargest(1,'Score')['Country'].values[0])

st.divider()

# create top bar chart
fig = px.bar(top, x='Score', y='Country', orientation='h',
             labels={'Score': 'Happiness Score (0–10)', 'Country': ''},
             template='plotly_white', title=f'Happiest {top_n} Countries in 2023 - {selected_region}')

fig.update_layout(title_font=dict(family='Arial', size=13, color='slategrey'),
                  xaxis=dict(range=[0, 8.0]))


st.plotly_chart(fig, width='stretch', height=600) 

st.divider()
st.caption("Built with Streamlit + Plotly")


---
## Class Exercise 💪 💻

1. Add another chart of your choice to the dashboard that uses a diverging continuous colour scale (something where values go above and below a meaningful midpoint). Label the midpoint in an annotation.

2. Publish your expanded dashboard through you classwork repo.


In [1]:
# Calculate global average
global_average = df['Score'].mean()

# Calculate difference from global average
df['Difference'] = df['Score'] - global_average

# Create chart
fig2 = px.bar(
    df.sort_values('Difference'),
    x='Difference',
    y='Country',
    orientation='h',
    color='Difference',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title='Happiness Score Compared with Global Average'
)

# Mark the midpoint
fig2.add_vline(
    x=0,
    line_dash='dash',
    line_color='black',
    annotation_text=f'Global Average = {global_average:.2f}'
)

fig2.show()

NameError: name 'df' is not defined

In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/world_happiness_2023.csv')

df.columns = [
    'Country', 'Region', 'Score', 'GDP',
    'Social_Support', 'Life_Expectancy',
    'Freedom', 'Generosity', 'Corruption'
]

df.head()

ModuleNotFoundError: No module named 'pandas'

In [3]:
%pip install pandas plotly

  Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl.metadata (16 kB)
Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl (10.7 MB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 646.5 kB/s eta 0:00:15
   -- ------------------------------------- 0.5/9.7 MB 646.5 kB/s eta 0:00:15
   -- ------------------------------------- 0.5/9.7 MB 646.5 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/9.7 MB 509.2 kB/s eta 0:00:18
   ---- ----------------------------------- 1.0/9.7 MB 629.2 kB/s eta 0:00:14
   ---- ----------------------------------- 1.0/9.7 MB 629.2 kB/s eta 0:00:14
   ----- ---------------------------------- 1.3/9.7 MB 680.6 kB/s eta 0:00:13
   ------ --------------------------------- 1


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import plotly.express as px

print("Pandas and Plotly installed successfully!")

Pandas and Plotly installed successfully!


In [4]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('world_happiness_2023.csv')

df.columns = [
    'Country', 'Region', 'Score', 'GDP',
    'Social_Support', 'Life_Expectancy',
    'Freedom', 'Generosity', 'Corruption'
]

df.head()

,Country,Region,Score,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption
0,Finland,Western Europe,7.804,10.775,0.954,71.9,0.949,0.142,0.179
1,Denmark,Western Europe,7.586,10.933,0.954,72.7,0.931,0.168,0.234
2,Iceland,Western Europe,7.525,10.878,0.983,72.5,0.961,0.260,0.150
3,Israel,Middle East and North Africa,7.473,10.527,0.916,72.4,0.903,0.149,0.826
4,Netherlands,Western Europe,7.464,11.015,0.939,72.4,0.879,0.240,0.296


In [6]:
# Calculate global average happiness score
global_average = df['Score'].mean()

# Calculate difference from global average
df['Difference'] = df['Score'] - global_average

# Select 10 happiest and 10 lowest-scoring countries
comparison = pd.concat([
    df.nsmallest(10, 'Score'),
    df.nlargest(10, 'Score')
])

comparison = comparison.sort_values('Difference')

# Create diverging colour chart
max_difference = comparison['Difference'].abs().max()

fig2 = px.bar(
    comparison,
    x='Difference',
    y='Country',
    orientation='h',
    color='Difference',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    range_color=[-max_difference, max_difference],
    title='Happiness Compared with Global Average',
    labels={
        'Difference': 'Difference from Global Average',
        'Country': 'Country'
    },
    template='plotly_white'
)

# Add global average annotation
fig2.add_vline(
    x=0,
    line_dash='dash',
    line_color='black',
    annotation_text=f'Global Average = {global_average:.2f}',
    annotation_position='top'
)

fig2.show()

In [8]:
import plotly.io as pio

pio.renderers.default = 'iframe'

fig2.show()